# Media Analytics – ETL Pipeline
### Traditional vs. Digital Media Consumption Shift
#### Business Questions:
1. How have viewership and subscriber metrics trended for traditional vs. digital platforms from 2015–2024?
2. When did the shift from traditional to digital media accelerate most significantly?
3. How does user engagement (hours, sessions, retention) compare across media types and regions?
4. How have streaming subscription prices changed over time, and how do pricing tiers relate to subscriber growth?
5. What factors most influence users to switch from traditional to digital media?

In [1448]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

---
### Part 1 – Data Sources Profile

| # | File | Rows | Feeds |
|---|---|---|---|
| 1 | `streaming_service.csv` | 777 | FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN |
| 2 | `platform_summary.csv` | 12 | DIM_PLATFORM |
| 3 | `platform_financials_comprehensive.csv` | 10 | FACT_MEDIA_PERFORMANCE |
| 4 | `industry_metrics.csv` | 9 | FACT_MEDIA_PERFORMANCE |
| 5 | `traditional_media_viewership_monthly.csv` | 1,624 | FACT_MEDIA_PERFORMANCE |
| 6 | `user_engagement_monthly.csv` | 4,120 | FACT_ENGAGEMENT |
| 7 | `switch_factor_survey.csv` | 2,025 | FACT_ENGAGEMENT, DIM_SWITCH_REASON |
| 8 | `platform_subscriber_monthly.csv` | 640 | FACT_MEDIA_PERFORMANCE |

### 1.1 – streaming_service.csv
**Source:** Kaggle | **Feeds:** FACT_SUBSCRIPTION_PRICING, DIM_SUBSCRIPTION_PLAN  
**Known issues:** Date stored as `Jul-2011` string format — needs parsing

In [1451]:
streaming_df = pd.read_csv('data sources/streaming_service.csv')
streaming_df.head(5)

,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99
3,Netflix,Oct-2011,7.99
4,Netflix,Nov-2011,7.99


In [1452]:
#shape -- rows/columns
print(f"Number of rows & columns: {streaming_df.shape[0]}, {streaming_df.shape[1]}")

#null count
print(f"\nTotal number of nulls in each column: \n{streaming_df.isnull().sum()}")

#unique services
print(f"\nUnique services within dataframe: {streaming_df['service'].unique()}")

#max/min dates
print(f"\nDate range (raw): {streaming_df['date'].min()} → {streaming_df['date'].max()}")

Number of rows & columns: 777, 3

Total number of nulls in each column: 
service    0
date       0
price      0
dtype: int64

Unique services within dataframe: ['Netflix' 'Disney+' 'HBO Max' 'Hulu' 'Paramount+' 'Prime Video'
 'Apple TV+' 'Peacock' 'Shudder' 'Crunchyroll']

Date range (raw): Apr-2012 → Sep-2023


### 1.2 – platform_summary.csv
**Source:** Streaming Platform Dataset | **Feeds:** DIM_PLATFORM  
**Known issues:** Wide table (36 cols), many NaN — only 5 dim cols needed

In [1454]:
platform_df = pd.read_csv('data sources/Streaming Platform Dataset/platform_summary.csv')
platform_df.head(5)

,platform,launch_year,parent_company,content_focus,business_model,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions,quarterly_eps_usd,...,projected_2026_subscribers_millions,sports_rights_spend_2026_usd_millions,sports_rights_global_share_pct,streaming_revenue_q4_usd_millions,streaming_revenue_growth_pct,monthly_streams_millions,streams_growth_pct,monthly_hours_watched_millions,monthly_hours_growth_pct,peak_concurrent_viewers_millions
0,Netflix,2007,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2026-01-19,325.0,12050.0,2410.0,0.56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Disney+,2019,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2026-02-07,126.0,NaN,NaN,NaN,...,284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hulu,2008,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazon Prime Video,2006,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,3800.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Paramount+,2021,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,1136.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1455]:
#shape before dim colum extract
print(f"Number of rows & columns (before): {platform_df.shape[0]}, {platform_df.shape[1]}")

#only need 5 dim columns -- for DIM_PLATFORM -- so deriving those
platform_df = platform_df[['platform', 'parent_company', 'business_model', 'content_focus', 'launch_year']]
display(platform_df.head(5))

#shape before dim colum extract
print(f"Number of rows & columns (after): {platform_df.shape[0]}, {platform_df.shape[1]}")

Number of rows & columns (before): 12, 36


,platform,parent_company,business_model,content_focus,launch_year
0,Netflix,Netflix Inc.,Subscription + Ads,"Original series, films, licensed content",2007
1,Disney+,The Walt Disney Company,Subscription + Ads,"Family, Marvel, Star Wars, National Geographic",2019
2,Hulu,The Walt Disney Company,Subscription + Ads,"TV shows, next-day broadcast",2008
3,Amazon Prime Video,Amazon.com Inc.,Prime bundle + Subscription,"Original series, movies, sports",2006
4,Paramount+,Paramount Global,Subscription + Ads,"Nickelodeon, CBS, Showtime",2021


Number of rows & columns (after): 12, 5


In [1456]:
#null counts
print(f"\nTotal number of nulls in each column: \n{platform_df.isnull().sum()}")

#unique platforms
print(f"\nUnique platforms within dataframe: {platform_df['platform'].unique()}")
print(f"\nUnique parent companies within dataframe: {platform_df['parent_company'].unique()}")


Total number of nulls in each column: 
platform          0
parent_company    0
business_model    0
content_focus     0
launch_year       0
dtype: int64

Unique platforms within dataframe: ['Netflix' 'Disney+' 'Hulu' 'Amazon Prime Video' 'Paramount+' 'Peacock'
 'Apple TV+' 'Max' 'Roku' 'YouTube TV' 'Tubi' 'Pluto TV']

Unique parent companies within dataframe: ['Netflix Inc.' 'The Walt Disney Company' 'Amazon.com Inc.'
 'Paramount Global' 'Comcast' 'Apple Inc.' 'Warner Bros. Discovery'
 'Roku Inc.' 'Google' 'Fox Corporation']


### 1.3 – platform_financials_comprehensive.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** 10 rows only (1 per platform, snapshot); many sparse columns

In [1458]:
platform_fin_df = pd.read_csv('data sources/Streaming Platform Dataset/platform_financials_comprehensive.csv')
platform_fin_df.head(5)

,platform,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions,quarterly_eps_usd,annual_content_spend_usd_millions,annual_content_spend_growth_pct,ad_revenue_2025_usd_millions,ad_revenue_2026_projection_usd_millions,...,projected_2026_subscribers_millions,sports_rights_spend_2026_usd_millions,sports_rights_global_share_pct,streaming_revenue_q4_usd_millions,streaming_revenue_growth_pct,monthly_streams_millions,streams_growth_pct,monthly_hours_watched_millions,monthly_hours_growth_pct,peak_concurrent_viewers_millions
0,Netflix,2026-01-19,325.0,12050.0,2410.0,0.56,20000.0,10.0,1500.0,3000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Roku,2026-02-11,98.0,1395.0,80.5,0.53,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Disney+,2026-02-07,126.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazon Prime Video,2026-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,3800.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Paramount+,2026-02-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,1136.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1459]:
#only taking key financial columns relevant to the fact
platform_fin_df = platform_fin_df[['platform', 'reporting_date', 'subscribers_millions',
                                   'quarterly_revenue_usd_millions', 'quarterly_profit_usd_millions']]
platform_fin_df.head(5)

,platform,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions
0,Netflix,2026-01-19,325.0,12050.0,2410.0
1,Roku,2026-02-11,98.0,1395.0,80.5
2,Disney+,2026-02-07,126.0,NaN,NaN
3,Amazon Prime Video,2026-02-06,NaN,NaN,NaN
4,Paramount+,2026-02-06,NaN,NaN,NaN


In [1460]:
print('Shape:', platform_fin_df.shape)
print('Platforms:', list(platform_fin_df['platform']))
print('\nNull counts (key cols):')
print(platform_fin_df.isnull().sum())

Shape: (10, 5)
Platforms: ['Netflix', 'Roku', 'Disney+', 'Amazon Prime Video', 'Paramount+', 'YouTube TV', 'AMC Networks', 'ITVX', 'DAZN', 'Twitch']

Null counts (key cols):
platform                          0
reporting_date                    0
subscribers_millions              6
quarterly_revenue_usd_millions    7
quarterly_profit_usd_millions     8
dtype: int64


### 1.4 – platform_subscriber_monthly.csv
**Source:** Synthetic (AI-generated, anchored to real quarterly figures) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Null `revenue_usd_millions` for some platforms; seasonal churn spikes built in

In [1462]:
platform_subs_df = pd.read_csv('data sources/platform_subscriber_monthly.csv')

print('Rows & Columns:', platform_subs_df.shape)
print('Columns:', list(platform_subs_df.columns))

print('\nNull counts:')
print(platform_subs_df.isnull().sum())

print('\nSample rows:')
display(platform_subs_df.head(5))

print('\nPlatforms covered:', platform_subs_df['platform_name'].unique())
print('Date range:', platform_subs_df['year_month'].min(), '→', platform_subs_df['year_month'].max())

Rows & Columns: (640, 8)
Columns: ['year_month', 'platform_name', 'subscribers_millions', 'revenue_usd_millions', 'churn_rate_pct', 'new_subscribers_millions', 'cancelled_subscribers_millions', 'country_region']

Null counts:
year_month                          0
platform_name                       0
subscribers_millions                0
revenue_usd_millions              344
churn_rate_pct                      0
new_subscribers_millions            0
cancelled_subscribers_millions      0
country_region                      0
dtype: int64

Sample rows:


,year_month,platform_name,subscribers_millions,revenue_usd_millions,churn_rate_pct,new_subscribers_millions,cancelled_subscribers_millions,country_region
0,2015-01,Netflix,65.32,729.9,0.0596,0.73,0.32,US
1,2015-02,Netflix,64.85,719.2,0.0393,0.74,0.21,US
2,2015-03,Netflix,64.69,701.7,0.0416,0.48,0.22,US
3,2015-04,Netflix,65.16,700.4,0.0343,0.29,0.19,US
4,2015-05,Netflix,64.34,688.3,0.0409,0.42,0.22,US



Platforms covered: ['Netflix' 'Hulu' 'Amazon Prime Video' 'Disney+' 'Max' 'Apple TV+'
 'Peacock' 'Paramount+']
Date range: 2015-01 → 2024-12


### 1.5 – industry_metrics.csv
**Source:** Streaming Platform Dataset | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Annual granularity only; industry-level aggregates, not platform-level

In [1464]:
industry_metrics_df = pd.read_csv('data sources/Streaming Platform Dataset/industry_metrics.csv')

print('Rows & Columns:', industry_metrics_df.shape)
print('\nAll rows:')
display(industry_metrics_df.head(5))
print('\nNull counts:')
print(industry_metrics_df.isnull().sum())

Rows & Columns: (9, 8)

All rows:


,metric_category,metric_name,value_usd_billions,unit,year,source,value_pct,value_usd_millions
0,Global Content Spending,Total Global Content Spend 2025,245.0,USD Billions,2025,Ampere Analysis [citation:7],NaN,NaN
1,Global Content Spending,Total Global Content Spend 2026,255.0,USD Billions,2026,Ampere Analysis [citation:7],NaN,NaN
2,Global Content Spending,Streamer Content Spend 2026,101.0,USD Billions,2026,Ampere Analysis [citation:7],NaN,NaN
3,Global Content Spending,Streamer Share of Global Spend,NaN,Percentage,2026,Ampere Analysis [citation:7],40.0,NaN
4,Sports Rights,Global Streaming Sports Rights Spend 2025,13.2,USD Billions,2025,Ampere Analysis [citation:3],NaN,NaN



Null counts:
metric_category       0
metric_name           0
value_usd_billions    4
unit                  0
year                  0
source                0
value_pct             7
value_usd_millions    7
dtype: int64


### 1.6 – traditional_media_viewership_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_MEDIA_PERFORMANCE  
**Known issues:** Mixed date formats (`Jan-2010`, `2010/01`, `March 2010`), some `metric_value` stored as strings (`"13.6M"`), inconsistent `media_type` casing, 25 intentional duplicate rows

In [1466]:
traditional_media_df = pd.read_csv('data sources/traditional_media_viewership_monthly.csv')

print('Rows & Columns:', traditional_media_df.shape)
print('Columns:', list(traditional_media_df.columns))

print('\nmedia_type unique values:', traditional_media_df['media_type'].unique())

print('\nNull counts:')
print(traditional_media_df.isnull().sum())
print('\nSample rows:')
display(traditional_media_df.head(5))

Rows & Columns: (1624, 7)
Columns: ['report_month', 'platform_name', 'media_type', 'metric_name', 'metric_value', 'market', 'source']

media_type unique values: ['Broadcast TV' 'Cable TV' 'cable tv' 'CABLE TV' 'Print' 'print' 'Radio']

Null counts:
report_month     0
platform_name    0
media_type       0
metric_name      0
metric_value     0
market           0
source           0
dtype: int64

Sample rows:


,report_month,platform_name,media_type,metric_name,metric_value,market,source
0,Jan-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.63,US,Nielsen
1,Feb-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.42,US,Nielsen
2,March 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.6,Global,Nielsen
3,Apr-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.8M,US,Nielsen
4,May-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.28,US,Nielsen


### 1.7 – switch_factor_survey.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT, DIM_SWITCH_REASON  
**Known issues:** Inconsistent `switched_from` values (`Cable` vs `Cable TV` vs `cable tv`), many null `secondary_switch_reason`, duplicate `respondent_id` across survey years (re-contact design), mixed `satisfaction_score` types

In [1468]:
survey_df = pd.read_csv('data sources/switch_factor_survey.csv')

print('Shape:', survey_df.shape)
print('Columns:', list(survey_df.columns))

print('\nNull counts:')
print(survey_df.isnull().sum())
print('\nSample rows:')
display(survey_df.head(5))

#respondents per survey year
print('\nTotal Respondents per Survey Year:')
print(survey_df.groupby('survey_year')['respondent_id'].count())

Shape: (2025, 10)
Columns: ['survey_year', 'respondent_id', 'switched_from', 'switched_to', 'primary_switch_reason', 'secondary_switch_reason', 'age_group', 'region', 'household_income_bracket', 'satisfaction_score']

Null counts:
survey_year                    0
respondent_id                  0
switched_from                  0
switched_to                    0
primary_switch_reason          0
secondary_switch_reason     1105
age_group                      0
region                         0
household_income_bracket       0
satisfaction_score           173
dtype: int64

Sample rows:


,survey_year,respondent_id,switched_from,switched_to,primary_switch_reason,secondary_switch_reason,age_group,region,household_income_bracket,satisfaction_score
0,2015,10228,Broadcast TV,YouTube,Convenience,NaN,18-24,Midwest,$75K-$150K,6.89
1,2015,10051,Broadcast TV,Spotify,Content Selection,Recommendation,55+,South,$75K-$150K,8.00
2,2015,11518,newspaper,Amazon Prime Video,Device Availability,Device Availability,35-44,West,>$150K,8.00
3,2015,10563,Cable,Netflix,Price,NaN,18-24,South,<$35K,8.00
4,2015,10501,Cable,Max,Device Availability,NaN,55+,South,>$150K,6.89



Total Respondents per Survey Year:
survey_year
2015    220
2016    200
2017    216
2018    200
2019    207
2020    199
2021    194
2022    185
2023    203
2024    201
Name: respondent_id, dtype: int64


### 1.8 – user_engagement_monthly.csv
**Source:** Synthetic (AI-generated) | **Feeds:** FACT_ENGAGEMENT  
**Known issues:** Mixed date formats (`2015-01` vs `01/2015`), `retention_rate_pct` stored as decimal in some rows and `"78%"` string in others, nulls for new platforms in early months, 40 intentional duplicate rows

In [1470]:
user_engagement_df = pd.read_csv('data sources/user_engagement_monthly.csv')

print('Rows & Columns:', user_engagement_df.shape)

print('\nNull counts:')
print(user_engagement_df.isnull().sum())

print('\nSample rows:')
display(user_engagement_df.head(5))

print('\nMixed date formats (sample):', user_engagement_df['year_month'].unique()[:10])
print('\nretention_rate_pct mixed types (sample):')
print(user_engagement_df['retention_rate_pct'].head(20).tolist())

Rows & Columns: (4120, 8)

Null counts:
year_month                         0
platform_name                      0
media_type                         0
avg_hours_per_user_per_month       0
monthly_active_users_millions      0
avg_sessions_per_user_per_month    0
retention_rate_pct                 0
region                             0
dtype: int64

Sample rows:


,year_month,platform_name,media_type,avg_hours_per_user_per_month,monthly_active_users_millions,avg_sessions_per_user_per_month,retention_rate_pct,region
0,01/2015,Netflix,Streaming,32.75,64.72,9.1,0.9165,North America
1,2015-01,Netflix,Streaming,32.75,64.72,9.1,0.9165,Europe
2,2015-01,Netflix,Streaming,32.75,64.72,9.1,0.9165,APAC
3,03/2015,Netflix,Streaming,31.90,65.18,8.8,0.9268,North America
4,2015-03,Netflix,Streaming,31.90,65.18,8.8,92.7%,Europe



Mixed date formats (sample): ['01/2015' '2015-01' '03/2015' '2015-03' '2015-04' '2015-06' '06/2015'
 '2015-07' '2015-08' '09/2015']

retention_rate_pct mixed types (sample):
['0.9165', '0.9165', '0.9165', '0.9268', '92.7%', '0.9268', '0.9069', '90.7%', '0.9069', '0.8953', '89.5%', '0.8953', '0.9039', '0.9039', '0.9039', '0.9142', '0.9142', '0.9142', '0.9025', '90.2%']


---
### Part 2: Transformations
Clean and enrich raw data before building dims and facts.

| Block | Source | Transforms | Feeds |
|---|---|---|---|
| 2.1 | `streaming_service.csv` | T1–T6 | DIM_SUBSCRIPTION_PLAN, FACT_SUBSCRIPTION_PRICING |

In [1472]:
#removing duplicates -- no duplicates in the table
print(f"Rows before removing duplicates: {streaming_df.shape[0]}")
streaming_df = streaming_df.drop_duplicates()
print(f"Rows after removing duplicates: {streaming_df.shape[0]}")

Rows before removing duplicates: 777
Rows after removing duplicates: 777


In [1473]:
#standardizing date column
print("Before standardization:")
display(streaming_df[['service', 'date', 'price']].head(3))

print("\nAfter standardization:")
streaming_df['date'] = pd.to_datetime(streaming_df['date'], format='%b-%Y')
display(streaming_df[['service', 'date', 'price']].head(3))

Before standardization:


,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99



After standardization:


,service,date,price
0,Netflix,2011-07-01,7.99
1,Netflix,2011-08-01,7.99
2,Netflix,2011-09-01,7.99


In [1474]:
#price tier calculation
bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0-$7)', 'Mid ($7-$14)', 'Premium (>$14)']

streaming_df['price_tier'] = pd.cut(streaming_df['price'], bins=bins, labels = labels, right=True)
display(streaming_df[['service', 'date', 'price', 'price_tier']].head(3))
print('\nTier distribution:')
print(streaming_df['price_tier'].value_counts())

,service,date,price,price_tier
0,Netflix,2011-07-01,7.99,Mid ($7-$14)
1,Netflix,2011-08-01,7.99,Mid ($7-$14)
2,Netflix,2011-09-01,7.99,Mid ($7-$14)



Tier distribution:
price_tier
Mid ($7-$14)      560
Budget ($0-$7)    154
Premium (>$14)     63
Name: count, dtype: int64


In [1475]:
#price change mom = diff of `price` per service sorted by date
streaming_df = streaming_df.sort_values(['service', 'date'])

streaming_df['price_change_mom'] = (streaming_df.groupby('service')['price'].diff())

print('Months where price changed:')
price_changes = (
    streaming_df[(streaming_df['price_change_mom'] != 0) & streaming_df['price_change_mom'].notna()])

display(price_changes[['service', 'date', 'price', 'price_change_mom']].head(5))

Months where price changed:


,service,date,price,price_change_mom
588,Apple TV+,2022-10-01,6.99,2.0
603,Apple TV+,2024-01-01,9.99,3.0
167,Disney+,2021-03-01,7.99,1.0
188,Disney+,2022-12-01,10.99,3.0
198,Disney+,2023-10-01,13.99,3.0


In [1476]:
#cumulative price increase = current price minus the service's first recorded price
earliest = streaming_df.sort_values('date').groupby('service').first().reset_index()
latest = streaming_df.sort_values('date').groupby('service').last().reset_index()

first_last_df = pd.concat([earliest, latest])
first_last_df = first_last_df.sort_values('service').reset_index()

display(first_last_df.head(5))

,index,service,date,price,price_tier,price_change_mom
0,0,Apple TV+,2019-11-01,4.99,Budget ($0-$7),0.0
1,0,Apple TV+,2024-01-01,9.99,Mid ($7-$14),3.0
2,1,Crunchyroll,2020-08-01,7.99,Mid ($7-$14),0.0
3,1,Crunchyroll,2024-01-01,7.99,Mid ($7-$14),0.0
4,2,Disney+,2019-11-01,6.99,Budget ($0-$7),0.0


In [1477]:
streaming_df['cumulative_price_increase'] = (streaming_df.groupby('service')['price'].transform(lambda x: x - x.iloc[0]))
# streaming_df.head(5)
latest = streaming_df.sort_values('date').groupby('service').last().reset_index()
display(latest.head(5))

,service,date,price,price_tier,price_change_mom,cumulative_price_increase
0,Apple TV+,2024-01-01,9.99,Mid ($7-$14),3.0,5.0
1,Crunchyroll,2024-01-01,7.99,Mid ($7-$14),0.0,0.0
2,Disney+,2024-01-01,13.99,Mid ($7-$14),0.0,7.0
3,HBO Max,2024-01-01,15.99,Premium (>$14),0.0,1.0
4,Hulu,2024-01-01,17.99,Premium (>$14),3.0,6.0


In [1478]:
#service name normalization
name_map = {'HBO Max': 'Max', 'Prime Video' : 'Amazon Prime Video'}

streaming_df['platform'] = streaming_df['service'].replace(name_map)
print('Service → platform mapping:')
print(streaming_df[['service', 'platform']].drop_duplicates().to_string())

Service → platform mapping:
         service            platform
553    Apple TV+           Apple TV+
735  Crunchyroll         Crunchyroll
151      Disney+             Disney+
202      HBO Max                 Max
247         Hulu                Hulu
0        Netflix             Netflix
347   Paramount+          Paramount+
604      Peacock             Peacock
459  Prime Video  Amazon Prime Video
647      Shudder             Shudder


----
| Block | Source | Transforms | Feeds |
|---|---|---|---|
| 2.2 | `platform_summary.csv` | T7–T9 | DIM_PLATFORM |

In [1480]:
#removing duplicates -- no duplicates in the table
print(f"Rows before removing duplicates: {platform_df.shape[0]}")
platform_df = platform_df.drop_duplicates()
print(f"Rows after removing duplicates: {platform_df.shape[0]}")

Rows before removing duplicates: 12
Rows after removing duplicates: 12


In [1481]:
#deriving columns --> is_digital, media_sector
print('Before:')
display(platform_df.head(3))

platform_df = platform_df.copy()
platform_df['is_digital'] = True
platform_df['media_sector'] = 'Streaming'

print('\nAfter:')
display(platform_df.head(3))

Before:


,platform,parent_company,business_model,content_focus,launch_year
0,Netflix,Netflix Inc.,Subscription + Ads,"Original series, films, licensed content",2007
1,Disney+,The Walt Disney Company,Subscription + Ads,"Family, Marvel, Star Wars, National Geographic",2019
2,Hulu,The Walt Disney Company,Subscription + Ads,"TV shows, next-day broadcast",2008



After:


,platform,parent_company,business_model,content_focus,launch_year,is_digital,media_sector
0,Netflix,Netflix Inc.,Subscription + Ads,"Original series, films, licensed content",2007,True,Streaming
1,Disney+,The Walt Disney Company,Subscription + Ads,"Family, Marvel, Star Wars, National Geographic",2019,True,Streaming
2,Hulu,The Walt Disney Company,Subscription + Ads,"TV shows, next-day broadcast",2008,True,Streaming


In [1482]:
#platform alignment w/cross-source
platforms_in_streaming_df = set(streaming_df['platform'].unique())
platforms_in_platform_df = set(platform_df['platform'].unique())

print('In streaming_df but NOT in platform_df (will add manually):')
print(sorted(platforms_in_streaming_df - platforms_in_platform_df))

print('\nIn platform_df but NOT in streaming_df (no pricing history):')
print(sorted(platforms_in_platform_df - platforms_in_streaming_df))

In streaming_df but NOT in platform_df (will add manually):
['Crunchyroll', 'Shudder']

In platform_df but NOT in streaming_df (no pricing history):
['Pluto TV', 'Roku', 'Tubi', 'YouTube TV']


In [1483]:
new_platforms_df = pd.DataFrame([
    {
        'platform': 'Crunchyroll', 'parent_company': 'Sony Group Corporation',
        'content_focus': 'Anime, manga', 'business_model': 'Subscription',
        'launch_year': 2009, 'is_digital': True, 'media_sector': 'Streaming',
    },
    {
        'platform': 'Shudder', 'parent_company': 'AMC Networks',
        'content_focus': 'Horror, thriller', 'business_model': 'Subscription',
        'launch_year': 2015, 'is_digital': True, 'media_sector': 'Streaming',
    },
])

platform_df_enriched = pd.concat([platform_df, new_platforms_df],ignore_index=True)

print(f'\nRows before enrichment: {len(platform_df)}')
print(f'Rows after enrichment:  {len(platform_df_enriched)}')
display(platform_df_enriched['platform'].unique())


Rows before enrichment: 12
Rows after enrichment:  14


array(['Netflix', 'Disney+', 'Hulu', 'Amazon Prime Video', 'Paramount+',
       'Peacock', 'Apple TV+', 'Max', 'Roku', 'YouTube TV', 'Tubi',
       'Pluto TV', 'Crunchyroll', 'Shudder'], dtype=object)

----
| Block | Source | Transforms | Feeds |
|---|---|---|---|
| 2.3 | `traditional_media_viewership + platform_subscriber + platform_financials` | T10–T16 | DIM_MEDIA_TYPE, DIM_GEOGRAPHY, FACT_MEDIA_PERFORMANCE |

In [1485]:
#removing duplicates
print(f"Tradition_Media -- Rows before removing duplicates: {len(traditional_media_df)}, Number of duplicates: {traditional_media_df.duplicated().sum()}")
print(f"Platforms_subs -- Rows before removing duplicates: {len(platform_subs_df)}, Number of duplicates: {platform_subs_df.duplicated().sum()}")
print(f"Platform_fin -- Rows before removing duplicates: {len(platform_fin_df)}, Number of duplicates: {platform_fin_df.duplicated().sum()}")

Tradition_Media -- Rows before removing duplicates: 1624, Number of duplicates: 25
Platforms_subs -- Rows before removing duplicates: 640, Number of duplicates: 0
Platform_fin -- Rows before removing duplicates: 10, Number of duplicates: 0


In [1486]:
#after removing duplicates in traditional_media_df
traditional_media_df = traditional_media_df.drop_duplicates().reset_index(drop=True)
print(f"Tradition_Media -- Rows before removing duplicates: {len(traditional_media_df)}")

Tradition_Media -- Rows before removing duplicates: 1599


In [1487]:
##### T11 – Standardize Mixed Date Formats (traditional_media.report_month) 
# Raw values arrive in 4+ formats: `Jan-2010`, `March 2010`, `2010/11`, `December 2010`. Parse all → `datetime` (month-anchored).
print('Before — sample report_month values:')
print(traditional_media_df['report_month'].drop_duplicates().head(10).tolist())

def parse_report_month(val):
    for fmt in ('%b-%Y', '%B %Y', '%Y/%m', '%Y-%m'):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(val, errors='coerce')

traditional_media_df['date'] = traditional_media_df['report_month'].apply(parse_report_month)
print(f'\nAfter  — unparsed rows: {traditional_media_df["date"].isna().sum()}')
print('Sample dates:', traditional_media_df['date'].drop_duplicates().head(5).tolist())
display(traditional_media_df.head(5))

Before — sample report_month values:
['Jan-2010', 'Feb-2010', 'March 2010', 'Apr-2010', 'May-2010', 'June 2010', 'July 2010', 'Aug-2010', 'Sep-2010', 'Oct-2010']

After  — unparsed rows: 0
Sample dates: [Timestamp('2010-01-01 00:00:00'), Timestamp('2010-02-01 00:00:00'), Timestamp('2010-03-01 00:00:00'), Timestamp('2010-04-01 00:00:00'), Timestamp('2010-05-01 00:00:00')]


,report_month,platform_name,media_type,metric_name,metric_value,market,source,date
0,Jan-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.63,US,Nielsen,2010-01-01
1,Feb-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.42,US,Nielsen,2010-02-01
2,March 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.6,Global,Nielsen,2010-03-01
3,Apr-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.8M,US,Nielsen,2010-04-01
4,May-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.28,US,Nielsen,2010-05-01


In [1488]:
#Collapses `cable tv` / `CABLE TV` / `Cable TV` → `Cable TV`, and `print` → `Print`.
#- `market`: `us` / `U.S.` / `US` → `US` 
print('Before — distinct media_type:')
print(sorted(traditional_media_df['media_type'].unique()))
print('Before — market:', sorted(traditional_media_df['market'].unique()))


traditional_media_df['media_type'] = (traditional_media_df['media_type'].str.strip().str.title()
                                    .replace({'Cable Tv': 'Cable TV', 'Broadcast Tv': 'Broadcast TV'}))

traditional_media_df['market'] = (traditional_media_df['market'].str.replace('.', '', regex=False)
        .str.strip().str.upper().replace({'US': 'US'}) )

traditional_media_df['market'] = traditional_media_df['market'].replace({'GLOBAL': 'Global'})

print('\nAfter  — distinct media_type:')
print(sorted(traditional_media_df['media_type'].unique()))

print('After  — market:', sorted(traditional_media_df['market'].unique()))

Before — distinct media_type:
['Broadcast TV', 'CABLE TV', 'Cable TV', 'Print', 'Radio', 'cable tv', 'print']
Before — market: ['Global', 'U.S.', 'US', 'us']

After  — distinct media_type:
['Broadcast TV', 'Cable TV', 'Print', 'Radio']
After  — market: ['Global', 'US']


In [1489]:
#some rows carry "13.8M" strings mixed in with numeric values. Strip suffix and cast to float.
suspect = traditional_media_df['metric_value'].str.contains('M', regex = False)
print('Sample:', traditional_media_df.loc[suspect, 'metric_value'].head(5).tolist())

traditional_media_df['metric_value'] = (traditional_media_df['metric_value'].str.rstrip('M'))
traditional_media_df['metric_value'] = (traditional_media_df['metric_value'].astype(float))

print(f'\nAfter — dtype: {traditional_media_df["metric_value"].dtype}, Sample: {traditional_media_df['metric_value'].head(5).tolist()}')


Sample: ['13.8M', '12.8M', '11.6M', '11.0M', '10.4M']

After — dtype: float64, Sample: [13.63, 13.42, 13.6, 13.8, 13.28]


In [1490]:
# Parse year_month (platform_subscriber) -- Source `year_month` is ISO-ish string (`2015-01`); parse to `datetime` and expose as `date`.
platform_subs_clean_df = platform_subs_df.copy()
platform_subs_clean_df['date'] = pd.to_datetime(platform_subs_clean_df['year_month'], format='%Y-%m')
print(f'Parsed {platform_subs_clean_df["date"].notna().sum()} / {len(platform_subs_clean_df)} rows')
print('Date range:', platform_subs_clean_df['date'].min(), '→', platform_subs_clean_df['date'].max())

Parsed 640 / 640 rows
Date range: 2015-01-01 00:00:00 → 2024-12-01 00:00:00


In [1491]:
#Year-over-year subscriber growth %: (subs_t − subs_t−12) / subs_t−12 × 100, per platform, sorted by date.
platform_subs_clean_df = platform_subs_clean_df.sort_values(['platform_name', 'date'])

platform_subs_clean_df['subs_lag_12'] = (platform_subs_clean_df.groupby('platform_name')['subscribers_millions'].shift(12))

platform_subs_clean_df['yoy_growth_pct'] = ((platform_subs_clean_df['subscribers_millions'] - platform_subs_clean_df['subs_lag_12'])
                                        / platform_subs_clean_df['subs_lag_12'] * 100).round(2)

print('Sample YoY values (Netflix, 2016+):')
display(platform_subs_clean_df[platform_subs_clean_df['platform_name'] == 'Netflix']
        [['date','subscribers_millions','subs_lag_12','yoy_growth_pct']].head(6))

Sample YoY values (Netflix, 2016+):


,date,subscribers_millions,subs_lag_12,yoy_growth_pct
0,2015-01-01,65.32,NaN,NaN
1,2015-02-01,64.85,NaN,NaN
2,2015-03-01,64.69,NaN,NaN
3,2015-04-01,65.16,NaN,NaN
4,2015-05-01,64.34,NaN,NaN
5,2015-06-01,65.95,NaN,NaN


In [1492]:
#Running cumulative of `new_subscribers_millions` per platform. Feeds `FACT_MEDIA_PERFORMANCE.cumulative_subscribers`.
platform_subs_clean_df['cumulative_subscribers'] = (platform_subs_clean_df.groupby('platform_name')['new_subscribers_millions']
                                                    .cumsum().round(2))

print('Sample cumulative values (Netflix):')
display(platform_subs_clean_df[platform_subs_clean_df['platform_name'] == 'Netflix']
        [['date','new_subscribers_millions','cumulative_subscribers']].head(6))

Sample cumulative values (Netflix):


,date,new_subscribers_millions,cumulative_subscribers
0,2015-01-01,0.73,0.73
1,2015-02-01,0.74,1.47
2,2015-03-01,0.48,1.95
3,2015-04-01,0.29,2.24
4,2015-05-01,0.42,2.66
5,2015-06-01,0.61,3.27


----
| Block | Source | Transforms | Feeds |
|---|---|---|---|
| 2.4 | `user_engagement + switch_factor_survey` | T17–T20 | DIM_SWITCH_REASON, FACT_ENGAGEMENT |

In [1494]:
print(f'Before: {len(user_engagement_df)} rows, {user_engagement_df.duplicated().sum()} duplicates')
user_engagement_df = user_engagement_df.drop_duplicates().reset_index(drop=True)
print(f'After : {len(user_engagement_df)} rows, {user_engagement_df.duplicated().sum()} duplicates')

Before: 4120 rows, 40 duplicates
After : 4080 rows, 0 duplicates


In [1495]:
#standardization of mixed date formats
print('Before — sample year_month values:')
print(user_engagement_df['year_month'].drop_duplicates().head(10).tolist())

def parse_ym(val): 
    for fmt in ('%Y-%m', '%m/%Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

user_engagement_df['date'] = user_engagement_df['year_month'].apply(parse_ym)
print(f'\nUnparsed rows: {user_engagement_df["date"].isna().sum()}')
print('Date range:', user_engagement_df['date'].min(), '→', user_engagement_df['date'].max())

Before — sample year_month values:
['01/2015', '2015-01', '03/2015', '2015-03', '2015-04', '2015-06', '06/2015', '2015-07', '2015-08', '09/2015']

Unparsed rows: 0
Date range: 2015-01-01 00:00:00 → 2024-12-01 00:00:00


In [1496]:
#normalizing retention_rate_pct
print('Before — sample raw values:')
print(user_engagement_df['retention_rate_pct'].dropna().head(10).tolist())

def to_pct(val):
    if pd.isna(val):
        return None
    s = str(val).strip().rstrip('%')
    try:
        f = float(s)
    except ValueError:
        return None
    # Decimal form (e.g. 0.9165) → 91.65; already-percent form (e.g. 92.7) stays
    return round(f * 100, 2) if f <= 1 else round(f, 2)

user_engagement_df['retention_rate_pct'] = user_engagement_df['retention_rate_pct'].apply(to_pct)
print('\nAfter — dtype:', user_engagement_df['retention_rate_pct'].dtype)
print('Sample normalized values:', user_engagement_df['retention_rate_pct'].dropna().head(10).tolist())
print('Min/Max:', user_engagement_df['retention_rate_pct'].min(), user_engagement_df['retention_rate_pct'].max())

Before — sample raw values:
['0.9165', '0.9165', '0.9165', '0.9268', '92.7%', '0.9268', '0.9069', '90.7%', '0.9069', '0.8953']

After — dtype: float64
Sample normalized values: [91.65, 91.65, 91.65, 92.68, 92.7, 92.68, 90.69, 90.7, 90.69, 89.53]
Min/Max: 42.21 99.0


In [1497]:
#normalizing switched_from column
print(f'Before — {survey_df["switched_from"].nunique()} distinct values:')
print(sorted(survey_df['switched_from'].dropna().unique()))

canonical_map = {
    # Cable family
    'Cable': 'Cable TV', 'cable tv': 'Cable TV', 'CABLE TV': 'Cable TV',
    'Cable TV': 'Cable TV', 'Cable television': 'Cable TV',
    # Broadcast family
    'Broadcast': 'Broadcast TV', 'broadcast tv': 'Broadcast TV',
    'Broadcast TV': 'Broadcast TV', 'Network TV': 'Broadcast TV',
    # Print family
    'Print': 'Print', 'print media': 'Print', 'Print Media': 'Print',
    'Newspaper': 'Print', 'newspaper': 'Print',
    # Satellite radio family
    'Satellite radio': 'Satellite Radio', 'satellite radio': 'Satellite Radio',
    'Satellite Radio': 'Satellite Radio', 'SiriusXM': 'Satellite Radio',
}

survey_df['switched_from'] = survey_df['switched_from'].map(canonical_map).fillna(survey_df['switched_from'])

print(f'\nAfter — {survey_df["switched_from"].nunique()} distinct values:')
print(sorted(survey_df['switched_from'].dropna().unique()))

Before — 18 distinct values:
['Broadcast', 'Broadcast TV', 'CABLE TV', 'Cable', 'Cable TV', 'Cable television', 'Network TV', 'Newspaper', 'Print', 'Print Media', 'Satellite Radio', 'Satellite radio', 'SiriusXM', 'broadcast tv', 'cable tv', 'newspaper', 'print media', 'satellite radio']

After — 4 distinct values:
['Broadcast TV', 'Cable TV', 'Print', 'Satellite Radio']


---
## Part 3 – Loading All Dimensions

Each dim includes a **delta demo** showing how its SCD type handles incoming change.

| # | Dimension | SCD Type | Delta Demo |
|---|---|---|---|
| 3.1 | `DIM_DATE` | SCD0 | Attempted update rejected (immutable) |
| 3.2 | `DIM_MEDIA_TYPE` | SCD1 | Sub-category correction ("Linear" → "Linear TV") |
| 3.3 | `DIM_GEOGRAPHY` | SCD1 | Region rename ("APAC" → "Asia Pacific") |
| 3.4 | `DIM_SWITCH_REASON` | SCD1 | Category correction ("Economic" → "Cost") |
| 3.5 | `DIM_SUBSCRIPTION_PLAN` | SCD2 | Netflix price increase ($15.49 → $17.99, Jan 2024) |
| 3.6 | `DIM_PLATFORM` | SCD3 | Max business model change (May 2023) |

#### 3.1 – DIM_DATE (SCD0)

Generated programmatically — monthly grain from 2010-01 through 2026-12 (covers all fact data).
**SCD0 rationale:** dates are immutable facts — `2020-03-01` will always be March 2020. No updates ever applied.
**Hierarchy (for drill-down):** `decade → year → quarter → month`

In [1500]:
# Build monthly date dimension
date_range = pd.date_range(start='2010-01-01', end='2026-12-01', freq='MS')

Dim_Date = pd.DataFrame({'full_date': date_range})
Dim_Date['year']        = Dim_Date['full_date'].dt.year
Dim_Date['quarter']     = 'Q' + Dim_Date['full_date'].dt.quarter.astype(str)
Dim_Date['month']       = Dim_Date['full_date'].dt.month
Dim_Date['month_name']  = Dim_Date['full_date'].dt.strftime('%B')
Dim_Date['decade']      = (Dim_Date['year'] // 10 * 10).astype(str) + 's'
Dim_Date['date_key']    = range(1, len(Dim_Date) + 1)

Dim_Date = Dim_Date[['date_key', 'full_date', 'year', 'quarter', 'month', 'month_name', 'decade']]

print(f'Dim_Date: {len(Dim_Date)} rows (monthly grain, {Dim_Date.year.min()}–{Dim_Date.year.max()})')
print('\nSample (first 5 + last 3 rows):')
display(pd.concat([Dim_Date.head(5), Dim_Date.tail(3)]))

Dim_Date: 204 rows (monthly grain, 2010–2026)

Sample (first 5 + last 3 rows):


,date_key,full_date,year,quarter,month,month_name,decade
0,1,2010-01-01,2010,Q1,1,January,2010s
1,2,2010-02-01,2010,Q1,2,February,2010s
2,3,2010-03-01,2010,Q1,3,March,2010s
3,4,2010-04-01,2010,Q2,4,April,2010s
4,5,2010-05-01,2010,Q2,5,May,2010s
201,202,2026-10-01,2026,Q4,10,October,2020s
202,203,2026-11-01,2026,Q4,11,November,2020s
203,204,2026-12-01,2026,Q4,12,December,2020s


##### SCD0 Delta Demo — attempted update is rejected

Simulates: source system wants to change `month_name` for `2020-03` from "March" to "Mar".  
**SCD0 = immutable.** Update is rejected; dim stays unchanged.

In [1502]:
# SCD0 guard: any attempted write to an immutable column is rejected
SCD0_IMMUTABLE_COLS = {'full_date', 'year', 'quarter', 'month', 'month_name', 'decade'}

def scd0_update(df, key_col, key_val, col, new_val):
    if col in SCD0_IMMUTABLE_COLS:
        print(f"[SCD0] REJECTED: '{col}' is immutable — update blocked.")
        return df  # unchanged
    df.loc[df[key_col] == key_val, col] = new_val
    print(f"[SCD0] APPLIED: {col} updated for {key_col}={key_val}.")
    return df

target_key = Dim_Date.loc[Dim_Date['full_date'] == '2020-03-01', 'date_key'].iloc[0]

print('BEFORE')
print(Dim_Date[Dim_Date['date_key'] == target_key][['date_key','full_date','month_name']].to_string(index=False))

# Actually attempt the update through the SCD0 guard
print('\nAttempting: change month_name for 2020-03 from "March" to "Mar"')
Dim_Date = scd0_update(Dim_Date, 'date_key', target_key, 'month_name', 'Mar')

print('\nAFTER — unchanged, guard rejected the write:')
print(Dim_Date[Dim_Date['date_key'] == target_key][['date_key','full_date','month_name']].to_string(index=False))


BEFORE
 date_key  full_date month_name
      123 2020-03-01      March

Attempting: change month_name for 2020-03 from "March" to "Mar"
[SCD0] REJECTED: 'month_name' is immutable — update blocked.

AFTER — unchanged, guard rejected the write:
 date_key  full_date month_name
      123 2020-03-01      March


#### 3.2 – DIM_MEDIA_TYPE (SCD1)

Derived from `traditional_media_df` + `user_engagement_df`. Normalizes inconsistent casing (`cable tv` / `CABLE TV` → `Cable TV`) and adds `category` + `sub_category`.

**SCD1 rationale:** media type taxonomy corrections (e.g. renaming a sub-category) overwrite in place — we don't retain prior labels.
**Diagram columns:** `media_type_key, media_type, category, sub_category`


In [1504]:
display(traditional_media_df.head(3))
display(user_engagement_df.head(3))

,report_month,platform_name,media_type,metric_name,metric_value,market,source,date
0,Jan-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.63,US,Nielsen,2010-01-01
1,Feb-2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.42,US,Nielsen,2010-02-01
2,March 2010,CBS Evening News,Broadcast TV,avg_viewers_millions,13.60,Global,Nielsen,2010-03-01


,year_month,platform_name,media_type,avg_hours_per_user_per_month,monthly_active_users_millions,avg_sessions_per_user_per_month,retention_rate_pct,region,date
0,01/2015,Netflix,Streaming,32.75,64.72,9.1,91.65,North America,2015-01-01
1,2015-01,Netflix,Streaming,32.75,64.72,9.1,91.65,Europe,2015-01-01
2,2015-01,Netflix,Streaming,32.75,64.72,9.1,91.65,APAC,2015-01-01


In [1505]:
media_types = sorted(set(traditional_media_df['media_type'].unique()) | set(user_engagement_df['media_type'].unique()))

Dim_Media_Type = pd.DataFrame({'media_type': media_types})

# category + sub_category (diagram)
traditional_types = {'Broadcast TV', 'Cable TV', 'Print', 'Radio'}
Dim_Media_Type['category'] = Dim_Media_Type['media_type'].apply(
    lambda x: 'Traditional' if x in traditional_types else 'Digital'
)

sub_category_map = {
    'Broadcast TV': 'Linear TV',
    'Cable TV'    : 'Linear TV',
    'Print'       : 'Print Media',
    'Radio'       : 'Audio Broadcast',
}
Dim_Media_Type['sub_category'] = Dim_Media_Type['media_type'].map(sub_category_map).fillna('On-Demand')

Dim_Media_Type['media_type_key'] = range(1, len(Dim_Media_Type) + 1)
Dim_Media_Type = Dim_Media_Type[['media_type_key', 'media_type', 'category', 'sub_category']]

print(f'Dim_Media_Type: {len(Dim_Media_Type)} rows')
display(Dim_Media_Type)

Dim_Media_Type: 5 rows


,media_type_key,media_type,category,sub_category
0,1,Broadcast TV,Traditional,Linear TV
1,2,Cable TV,Traditional,Linear TV
2,3,Print,Traditional,Print Media
3,4,Radio,Traditional,Audio Broadcast
4,5,Streaming,Digital,On-Demand


##### SCD1 Delta Demo — sub_category correction

Simulates: analyst flags `"Linear TV"` should be standardized to `"Linear Broadcast"` for consistency with a newer taxonomy.  
**SCD1 = overwrite in place.** Same row, same `media_type_key`, value changed. No history retained.

In [1507]:
print("Before SCD1:")
display(Dim_Media_Type[Dim_Media_Type['sub_category'] == 'Linear TV'])

mask = Dim_Media_Type['sub_category'] == 'Linear TV'
Dim_Media_Type.loc[mask, 'sub_category'] = 'Linear Broadcast'

print("After SCD1 Update:")
display(Dim_Media_Type[Dim_Media_Type['sub_category'] == 'Linear Broadcast'])

Before SCD1:


,media_type_key,media_type,category,sub_category
0,1,Broadcast TV,Traditional,Linear TV
1,2,Cable TV,Traditional,Linear TV


After SCD1 Update:


,media_type_key,media_type,category,sub_category
0,1,Broadcast TV,Traditional,Linear Broadcast
1,2,Cable TV,Traditional,Linear Broadcast


### 3.3 – DIM_GEOGRAPHY (SCD1)

Derived from region/market/country fields across 4 sources:
- `platform_subscriber_monthly.country_region` (US)
- `traditional_media_viewership.market` (US, Global)
- `user_engagement_monthly.region` (North America, Europe, APAC)
- `switch_factor_survey.region` (Midwest, South, West, Northeast)

**SCD1 rationale:** corrections overwrite in place (e.g. normalizing "APAC" → "Asia Pacific"). No history retained because there's no analytic value in tracking prior labels.

In [1509]:
# Consume cleaned frames from Part 2 (raw frames still carry pre-dedup/pre-parse noise)
geos = set()
geos.update(platform_subs_clean_df['country_region'].dropna().unique())
geos.update(traditional_media_df['market'].dropna().unique())
geos.update(user_engagement_df['region'].dropna().unique())
geos.update(survey_df['region'].dropna().unique())

# Classify by level
level_map = {
    'US'            : ('Country',      True),
    'Global'        : ('Global',       False),
    'North America' : ('Continent',    False),
    'Europe'        : ('Continent',    False),
    'APAC'          : ('Continent',    False),
    'Midwest'       : ('US Sub-Region', True),
    'South'         : ('US Sub-Region', True),
    'West'          : ('US Sub-Region', True),
    'Northeast'     : ('US Sub-Region', True),
}

Dim_Geography = pd.DataFrame([
    {'region_name': g, 'region_level': level_map.get(g, ('Unknown', False))[0],
     'is_us_related': level_map.get(g, ('Unknown', False))[1]}
    for g in sorted(geos)
])
Dim_Geography['geography_key'] = range(1, len(Dim_Geography) + 1)
Dim_Geography = Dim_Geography[['geography_key', 'region_name', 'region_level', 'is_us_related']]

print(f'Dim_Geography: {len(Dim_Geography)} rows')
display(Dim_Geography)

Dim_Geography: 10 rows


,geography_key,region_name,region_level,is_us_related
0,1,APAC,Continent,False
1,2,Europe,Continent,False
2,3,Global,Global,False
3,4,International,Unknown,False
4,5,Midwest,US Sub-Region,True
5,6,North America,Continent,False
6,7,Northeast,US Sub-Region,True
7,8,South,US Sub-Region,True
8,9,US,Country,True
9,10,West,US Sub-Region,True


In [1510]:
print('BEFORE SCD1 update:')
display(Dim_Geography[Dim_Geography['region_name'] == 'APAC'])

# SCD1: overwrite the value in place, same key
mask = Dim_Geography['region_name'] == 'APAC'
Dim_Geography.loc[mask, 'region_name'] = 'Asia Pacific'

print('AFTER — same geography_key, new region_name (no history)')
display(Dim_Geography[Dim_Geography['region_name'] == 'Asia Pacific'])

BEFORE SCD1 update:


,geography_key,region_name,region_level,is_us_related
0,1,APAC,Continent,False


AFTER — same geography_key, new region_name (no history)


,geography_key,region_name,region_level,is_us_related
0,1,Asia Pacific,Continent,False


### 3.4 – DIM_SWITCH_REASON (SCD1)

**SCD1 rationale:** reason-category corrections overwrite in place — we don't retain prior taxonomy labels.
**Diagram columns:** `reason_key, primary_reason, reason_category, is_cost_related, is_content_related`


In [1512]:
# Collect distinct reasons from primary + secondary columns (cleaned in T20)
reasons = (
    set(survey_df['primary_switch_reason'].dropna().unique()) |
    set(survey_df['secondary_switch_reason'].dropna().unique())
)

category_map = {
    'Price'               : 'Cost',
    'Content Selection'   : 'Content',
    'Convenience'         : 'User Experience',
    'Device Availability' : 'Technology',
    'Recommendation'      : 'Social',
}

Dim_Switch_Reason = pd.DataFrame({'primary_reason': sorted(reasons)})
Dim_Switch_Reason['reason_category'] = Dim_Switch_Reason['primary_reason'].map(category_map).fillna('Other')
Dim_Switch_Reason['is_cost_related'] = Dim_Switch_Reason['reason_category'] == 'Cost'
Dim_Switch_Reason['is_content_related'] = Dim_Switch_Reason['reason_category'] == 'Content'
Dim_Switch_Reason['reason_key'] = range(1, len(Dim_Switch_Reason) + 1)

Dim_Switch_Reason = Dim_Switch_Reason[['reason_key', 'primary_reason', 'reason_category','is_cost_related', 'is_content_related']]

print(f'DimSwitchReason: {len(Dim_Switch_Reason)} rows')
display(Dim_Switch_Reason)

DimSwitchReason: 6 rows


,reason_key,primary_reason,reason_category,is_cost_related,is_content_related
0,1,Bundling,Other,False,False
1,2,Content Selection,Content,False,True
2,3,Convenience,User Experience,False,False
3,4,Device Availability,Technology,False,False
4,5,Price,Cost,True,False
5,6,Recommendation,Social,False,False


### 3.5 – DIM_SUBSCRIPTION_PLAN (SCD2)

Collapse consecutive months at the same price per platform into one SCD2 record.  
Each price change gets a **new row** with its own surrogate key, `effective_date`, `end_date`, and `is_current`.

In [1514]:
# Carry price_tier forward from streaming_df (derived in T3)
pricing = streaming_df[['platform', 'date', 'price', 'price_tier']].copy()
pricing['price_tier'] = pricing['price_tier'].astype(str)  # avoid categorical cross-product in groupby
pricing = pricing.sort_values(['platform', 'date'])

# Detect price changes and group consecutive same-price rows
pricing['price_changed'] = pricing.groupby('platform')['price'].diff().ne(0)
pricing['price_group']   = pricing.groupby('platform')['price_changed'].cumsum()

# Collapse each same-price block into one SCD2 record
Dim_Subscription_Plan = (
    pricing
    .groupby(['platform', 'price_group', 'price', 'price_tier'])
    .agg(effective_date=('date', 'min'), end_date=('date', 'max'))
    .reset_index()
    .drop(columns='price_group')
)

# Mark is_current = latest record per platform; drop its end_date
latest_idx = Dim_Subscription_Plan.groupby('platform')['end_date'].idxmax()
Dim_Subscription_Plan['is_current'] = False
Dim_Subscription_Plan.loc[latest_idx, 'is_current'] = True
Dim_Subscription_Plan.loc[Dim_Subscription_Plan['is_current'], 'end_date'] = None

# Surrogate key
Dim_Subscription_Plan['plan_key'] = range(1, len(Dim_Subscription_Plan) + 1)

Dim_Subscription_Plan = Dim_Subscription_Plan[[
    'plan_key', 'platform', 'price', 'price_tier',
    'effective_date', 'end_date', 'is_current'
]]

n_total   = len(Dim_Subscription_Plan)
n_current = Dim_Subscription_Plan['is_current'].sum()
print(f'Dim_Subscription_Plan: {n_total} rows ({n_current} current, {n_total - n_current} historical)')
print('\nNetflix price history (all SCD2 rows):')
display(Dim_Subscription_Plan[Dim_Subscription_Plan['platform'] == 'Netflix'])

Dim_Subscription_Plan: 26 rows (10 current, 16 historical)

Netflix price history (all SCD2 rows):


,plan_key,platform,price,price_tier,effective_date,end_date,is_current
16,17,Netflix,7.99,Mid ($7-$14),2011-07-01,2018-12-01,False
17,18,Netflix,8.99,Mid ($7-$14),2019-01-01,2021-12-01,False
18,19,Netflix,9.99,Mid ($7-$14),2022-01-01,2023-09-01,False
19,20,Netflix,15.49,Premium (>$14),2023-10-01,NaT,True


##### SCD2 Delta Demo — Netflix price increase ($15.49 → $17.99, Jan 2024)

Simulates an incoming price change. Expire the current row, insert a new row with a new surrogate key.

In [1516]:
delta_date    = pd.Timestamp('2024-01-01')
new_price     = 17.99
new_tier      = 'Premium (>$14)'
platform_name = 'Netflix'

# Expiring the current row
mask = (Dim_Subscription_Plan['platform'] == platform_name) & (Dim_Subscription_Plan['is_current'] == True)
Dim_Subscription_Plan.loc[mask, 'end_date']   = delta_date - pd.Timedelta(days=1)
Dim_Subscription_Plan.loc[mask, 'is_current'] = False

# Inserting new row with new surrogate key
new_row = {
    'plan_key'      : int(Dim_Subscription_Plan['plan_key'].max()) + 1,
    'platform' : platform_name,
    'price'         : new_price,
    'price_tier'    : new_tier,
    'effective_date': delta_date,
    'end_date'      : None,
    'is_current'    : True
}
Dim_Subscription_Plan = pd.concat([Dim_Subscription_Plan, pd.DataFrame([new_row])], ignore_index=True)

print('\nAFTER — full Netflix price history preserved (new row inserted)')
display(Dim_Subscription_Plan[Dim_Subscription_Plan['platform'] == platform_name])


AFTER — full Netflix price history preserved (new row inserted)


,plan_key,platform,price,price_tier,effective_date,end_date,is_current
16,17,Netflix,7.99,Mid ($7-$14),2011-07-01,2018-12-01,False
17,18,Netflix,8.99,Mid ($7-$14),2019-01-01,2021-12-01,False
18,19,Netflix,9.99,Mid ($7-$14),2022-01-01,2023-09-01,False
19,20,Netflix,15.49,Premium (>$14),2023-10-01,2023-12-31,False
26,27,Netflix,17.99,Premium (>$14),2024-01-01,NaT,True


### 3.6 – DIM_PLATFORM (SCD3)

Parent company and rebrand changes are rare — only current vs. previous needed in the same row.  
SCD3-tracked: `current_business_model` / `previous_business_model`, `current_parent_company` / `previous_parent_company`


In [1518]:
# Consume the enriched platform frame from T9 — all platforms already aligned
Dim_Platform = platform_df_enriched.copy()

# Rename 'platform' to 'platform_name' for consistency
Dim_Platform = Dim_Platform.rename(columns={'platform': 'platform'})

# SCD3 columns — initial load: previous_ columns are NULL
Dim_Platform['current_parent_company'] = Dim_Platform['parent_company']
Dim_Platform['previous_parent_company'] = None
Dim_Platform['current_business_model'] = Dim_Platform['business_model']
Dim_Platform['previous_business_model'] = None
Dim_Platform['parent_company_change_date']= None

# Surrogate key
Dim_Platform['platform_key'] = range(1, len(Dim_Platform) + 1)

# Final column order — matches constellation diagram exactly (11 cols)
Dim_Platform = Dim_Platform[['platform_key', 'platform',
    'current_parent_company', 'previous_parent_company',
    'current_business_model', 'previous_business_model',
    'parent_company_change_date', 'content_focus', 'launch_year', 'is_digital', 'media_sector'
]]

display_cols = [
    'platform_key', 'platform',
    'current_parent_company', 'previous_parent_company',
    'current_business_model', 'previous_business_model',
    'parent_company_change_date', 'is_digital', 'launch_year'
]

print('Dim_Platform initial load (previous_ columns all NULL):')
display(Dim_Platform[display_cols])

Dim_Platform initial load (previous_ columns all NULL):


,platform_key,platform,current_parent_company,previous_parent_company,current_business_model,previous_business_model,parent_company_change_date,is_digital,launch_year
0,1,Netflix,Netflix Inc.,None,Subscription + Ads,None,None,True,2007
1,2,Disney+,The Walt Disney Company,None,Subscription + Ads,None,None,True,2019
2,3,Hulu,The Walt Disney Company,None,Subscription + Ads,None,None,True,2008
3,4,Amazon Prime Video,Amazon.com Inc.,None,Prime bundle + Subscription,None,None,True,2006
4,5,Paramount+,Paramount Global,None,Subscription + Ads,None,None,True,2021
5,6,Peacock,Comcast,None,Subscription + Ads,None,None,True,2020
6,7,Apple TV+,Apple Inc.,None,Subscription,None,None,True,2019
7,8,Max,Warner Bros. Discovery,None,Subscription + Ads,None,None,True,2020
8,9,Roku,Roku Inc.,None,AVOD + Transactional,None,None,True,2008
9,10,YouTube TV,Google,None,AVOD + Subscription,None,None,True,2017


---
## Part 4 – Build All Fact Tables

Build all fact DataFrames in-memory before any database load. Dimensions are already built (Part 3), so FKs can be resolved in-memory.


| # | Fact | Type | Grain | FKs |
|---|---|---|---|---|
| 4.1 | `FACT_SUBSCRIPTION_PRICING` | Transactional Snapshot | Platform × Month | date, platform, plan |
| 4.2 | `FACT_MEDIA_PERFORMANCE` | Cumulative | Platform × Month × Media Type | date, platform, media_type |
| 4.3 | `FACT_ENGAGEMENT` | Transactional Snapshot | Platform × Month × Region × Switch Reason | date, platform, geo, switch_reason |


### 4.1 – FACT_SUBSCRIPTION_PRICING

All source cleaning already done in Part 2.1. This cell resolves all 3 FKs and finalizes to diagram columns.

**Measures:** `price`, `price_change_mom`, `cumulative_increase`

In [1521]:
# Source frame (already cleaned in Part 2.1) ---
Fact_Subscription_Pricing = streaming_df[['platform', 'date', 'price', 'price_change_mom', 'cumulative_price_increase'
]].rename(columns={'cumulative_price_increase': 'cumulative_increase'}).copy()

# Resolve plan_key FK (filter by date validity window)
plan_lookup = Dim_Subscription_Plan[['plan_key', 'platform', 'price', 'effective_date', 'end_date']].copy()
plan_lookup['effective_date'] = pd.to_datetime(plan_lookup['effective_date'])
plan_lookup['end_date_fill']  = pd.to_datetime(plan_lookup['end_date']).fillna(pd.Timestamp('2099-12-31'))

matched = Fact_Subscription_Pricing.merge(
    plan_lookup[['plan_key', 'platform', 'price', 'effective_date', 'end_date_fill']],
    on=['platform', 'price'], how='left'
)
matched = matched[(matched['date'] >= matched['effective_date']) &
    (matched['date'] <= matched['end_date_fill'])
].drop_duplicates(subset=['platform', 'date'])

Fact_Subscription_Pricing = Fact_Subscription_Pricing.merge(
    matched[['platform', 'date', 'plan_key']], on=['platform', 'date'], how='left'
)

#  Resolve platform_key FK 
Fact_Subscription_Pricing = Fact_Subscription_Pricing.merge(Dim_Platform[['platform_key', 'platform']], on='platform', how='left')

#  Resolve date_key FK 
Fact_Subscription_Pricing = Fact_Subscription_Pricing.merge(Dim_Date[['date_key', 'full_date']].rename(columns={'full_date': 'date'}), on='date', how='left')

#  Surrogate key + final column order (diagram-aligned, 7 cols) 
Fact_Subscription_Pricing['pricing_key'] = range(1, len(Fact_Subscription_Pricing) + 1)
Fact_Subscription_Pricing = Fact_Subscription_Pricing[['pricing_key', 'date_key', 'platform_key', 'plan_key',
    'price', 'price_change_mom', 'cumulative_increase']]

print(f'Fact_Subscription_Pricing: {len(Fact_Subscription_Pricing)} rows \u00d7 {Fact_Subscription_Pricing.shape[1]} cols')
print('Unresolved FKs:', Fact_Subscription_Pricing[['date_key','platform_key','plan_key']].isna().sum().to_dict())
print('\nFinal table (first 10 rows):')
display(Fact_Subscription_Pricing.head(10))

Fact_Subscription_Pricing: 777 rows × 7 cols
Unresolved FKs: {'date_key': 0, 'platform_key': 0, 'plan_key': 1}

Final table (first 10 rows):


,pricing_key,date_key,platform_key,plan_key,price,price_change_mom,cumulative_increase
0,1,119,7,3.0,4.99,NaN,0.0
1,2,120,7,3.0,4.99,0.0,0.0
2,3,121,7,3.0,4.99,0.0,0.0
3,4,122,7,3.0,4.99,0.0,0.0
4,5,123,7,3.0,4.99,0.0,0.0
5,6,124,7,3.0,4.99,0.0,0.0
6,7,125,7,3.0,4.99,0.0,0.0
7,8,126,7,3.0,4.99,0.0,0.0
8,9,127,7,3.0,4.99,0.0,0.0
9,10,128,7,3.0,4.99,0.0,0.0


### 4.2 – FACT_MEDIA_PERFORMANCE

Cumulative fact — unions two source halves:
- **Streaming half** (`PlatformSubscriberClean_df`): has subscribers / revenue / yoy / cumulative; `media_type` = `Streaming`.
- **Traditional half** (`traditional_media_df`): has viewership only; `media_type` = the source's media_type.

**Measures:** `subscribers_millions`, `viewership_millions`, `revenue_usd_millions`, `yoy_growth_pct`, `cumulative_subscribers`

In [1523]:
# Streaming half: from platform_subscriber 
streaming_half = platform_subs_clean_df[[
    'platform_name', 'date',
    'subscribers_millions', 'revenue_usd_millions',
    'yoy_growth_pct', 'cumulative_subscribers'
]].rename(columns={'platform_name': 'platform'}).copy()
streaming_half['media_type'] = 'Streaming'
streaming_half['viewership_millions'] = None

# Traditional half: from traditional_media (viewership only) 
trad = traditional_media_df[
    traditional_media_df['metric_name'] == 'avg_viewers_millions'
][['platform_name', 'date', 'media_type', 'metric_value']].rename(columns={
    'platform_name': 'platform',
    'metric_value' : 'viewership_millions',
}).copy()
trad['subscribers_millions']   = None
trad['revenue_usd_millions']   = None
trad['yoy_growth_pct']         = None
trad['cumulative_subscribers'] = None

# Union
Fact_Media_Performance = pd.concat([streaming_half, trad], ignore_index=True, sort=False)

# Resolve FKs 
Fact_Media_Performance = Fact_Media_Performance.merge(
    Dim_Platform[['platform_key', 'platform']], on='platform', how='left'
)
Fact_Media_Performance = Fact_Media_Performance.merge(
    Dim_Date[['date_key', 'full_date']].rename(columns={'full_date': 'date'}), on='date', how='left'
)
Fact_Media_Performance = Fact_Media_Performance.merge(
    Dim_Media_Type[['media_type_key', 'media_type']], on='media_type', how='left'
)

# Surrogate key + final column order (diagram-aligned)
Fact_Media_Performance['perf_key'] = range(1, len(Fact_Media_Performance) + 1)
Fact_Media_Performance = Fact_Media_Performance[[
    'perf_key', 'date_key', 'platform_key', 'media_type_key',
    'subscribers_millions', 'viewership_millions', 'revenue_usd_millions',
    'yoy_growth_pct', 'cumulative_subscribers'
]]

print(f'Fact_Media_Performance: {len(Fact_Media_Performance)} rows \u00d7 {Fact_Media_Performance.shape[1]} cols')
print('Unresolved FKs:', Fact_Media_Performance[['date_key','platform_key','media_type_key']].isna().sum().to_dict())
print('\nFinal table (first 10 rows):')
display(Fact_Media_Performance.head(10))


Fact_Media_Performance: 1533 rows × 9 cols
Unresolved FKs: {'date_key': 0, 'platform_key': 893, 'media_type_key': 0}

Final table (first 10 rows):


,perf_key,date_key,platform_key,media_type_key,subscribers_millions,viewership_millions,revenue_usd_millions,yoy_growth_pct,cumulative_subscribers
0,1,61,4.0,5,40.26,NaN,NaN,NaN,0.43
1,2,62,4.0,5,40.28,NaN,NaN,NaN,0.88
2,3,63,4.0,5,39.73,NaN,NaN,NaN,1.23
3,4,64,4.0,5,39.43,NaN,NaN,NaN,1.50
4,5,65,4.0,5,40.34,NaN,NaN,NaN,1.80
5,6,66,4.0,5,39.24,NaN,NaN,NaN,1.97
6,7,67,4.0,5,39.37,NaN,NaN,NaN,2.40
7,8,68,4.0,5,40.17,NaN,NaN,NaN,2.72
8,9,69,4.0,5,40.00,NaN,NaN,NaN,3.16
9,10,70,4.0,5,40.35,NaN,NaN,NaN,3.49


### 4.3 – FACT_ENGAGEMENT

Transactional Snapshot — one row per (platform \u00d7 month \u00d7 region). `switch_reason_key` is assigned from the modal `primary_switch_reason` for users who switched **to** this platform in this year/region (from the survey).

**Measures:** `avg_hours_per_user`, `monthly_active_users_millions`, `avg_sessions_per_month`, `retention_rate_pct`

In [1525]:
Fact_Engagement = user_engagement_df[['platform_name', 'date', 'region',
    'avg_hours_per_user_per_month',
    'monthly_active_users_millions',
    'avg_sessions_per_user_per_month',
    'retention_rate_pct'
]].rename(columns={
    'platform_name'                  : 'platform',
    'avg_hours_per_user_per_month'   : 'avg_hours_per_user',
    'avg_sessions_per_user_per_month': 'avg_sessions_per_month',
}).copy()

# Derive switch_reason_key: modal primary_switch_reason per (year, switched_to, region)
reason_counts = (
    survey_df
        .groupby(['survey_year', 'switched_to', 'region', 'primary_switch_reason'])
        .size().reset_index(name='n')
        .sort_values('n', ascending=False)
        .drop_duplicates(subset=['survey_year', 'switched_to', 'region'])
        .rename(columns={
            'switched_to'          : 'platform',
            'primary_switch_reason': 'primary_reason',
            'survey_year'          : 'year',
        })[['year', 'platform', 'region', 'primary_reason']]
)

Fact_Engagement['year'] = Fact_Engagement['date'].dt.year
Fact_Engagement = Fact_Engagement.merge(reason_counts, on=['year', 'platform', 'region'], how='left')

Fact_Engagement = Fact_Engagement.merge(
    Dim_Switch_Reason[['reason_key', 'primary_reason']], on='primary_reason', how='left'
).rename(columns={'reason_key': 'switch_reason_key'})

# Resolve remaining FKs 
Fact_Engagement = Fact_Engagement.merge(
    Dim_Platform[['platform_key', 'platform']], on='platform', how='left'
)
Fact_Engagement = Fact_Engagement.merge(
    Dim_Date[['date_key', 'full_date']].rename(columns={'full_date': 'date'}), on='date', how='left'
)
Fact_Engagement = Fact_Engagement.merge(
    Dim_Geography[['geography_key', 'region_name']].rename(columns={'region_name': 'region'}),
    on='region', how='left'
).rename(columns={'geography_key': 'geo_key'})

# Surrogate key + final column order (diagram-aligned)
Fact_Engagement['engagement_key'] = range(1, len(Fact_Engagement) + 1)
Fact_Engagement = Fact_Engagement[[
    'engagement_key', 'date_key', 'platform_key', 'geo_key', 'switch_reason_key',
    'avg_hours_per_user', 'monthly_active_users_millions',
    'avg_sessions_per_month', 'retention_rate_pct'
]]

print(f'Fact_Engagement: {len(Fact_Engagement)} rows × {Fact_Engagement.shape[1]} cols')
print('Unresolved FKs:', Fact_Engagement[['date_key','platform_key','geo_key','switch_reason_key']].isna().sum().to_dict())
print('\nFinal table (first 10 rows):')
display(Fact_Engagement.head(10))

Fact_Engagement: 4080 rows × 9 cols
Unresolved FKs: {'date_key': 0, 'platform_key': 2118, 'geo_key': 1360, 'switch_reason_key': 4080}

Final table (first 10 rows):


,engagement_key,date_key,platform_key,geo_key,switch_reason_key,avg_hours_per_user,monthly_active_users_millions,avg_sessions_per_month,retention_rate_pct
0,1,61,1.0,6.0,NaN,32.75,64.72,9.1,91.65
1,2,61,1.0,2.0,NaN,32.75,64.72,9.1,91.65
2,3,61,1.0,NaN,NaN,32.75,64.72,9.1,91.65
3,4,63,1.0,6.0,NaN,31.90,65.18,8.8,92.68
4,5,63,1.0,2.0,NaN,31.90,65.18,8.8,92.70
5,6,63,1.0,NaN,NaN,31.90,65.18,8.8,92.68
6,7,64,1.0,6.0,NaN,31.68,67.06,8.3,90.69
7,8,64,1.0,2.0,NaN,31.68,67.06,8.3,90.70
8,9,64,1.0,NaN,NaN,31.68,67.06,8.3,90.69
9,10,66,1.0,6.0,NaN,33.00,62.80,8.3,89.53


---
## Part 5 – Load Everything to PostgreSQL (`media` schema)

Load order: connect \u2192 dimensions \u2192 facts. Dimensions first because facts carry FKs into them.

**Targets:**
- `media.dim_subscription_plan` (SCD2)
- `media.dim_platform` (SCD3)
- `media.dim_date` (SCD0)
- `media.dim_media_type` (SCD1)
- `media.dim_geography` (SCD1)
- `media.dim_switch_reason` (SCD1)
- `media.fact_subscription_pricing`
- `media.fact_media_performance`
- `media.fact_engagement`

In [1579]:
import psycopg2
from psycopg2.extras import execute_values

conn = psycopg2.connect(
    host     = 'xx',
    database = 'xx',
    user     = 'xx',
    password = 'xx'
)
cursor = conn.cursor()

cursor.execute('CREATE SCHEMA IF NOT EXISTS "media"')
conn.commit()

print("Connected to cs689 | schema media ready")

Connected to cs689 | schema media ready


##### Load DIM_DATE (SCD0)

In [1581]:
cursor.execute('''
    DROP TABLE IF EXISTS "media".dim_date CASCADE;
    CREATE TABLE "media".dim_date (
        date_key   INTEGER PRIMARY KEY,
        full_date  DATE,
        year       INTEGER,
        quarter    TEXT,
        month      INTEGER,
        month_name TEXT,
        decade     TEXT
    );
''')
conn.commit()

d_load = Dim_Date.copy()
d_load['full_date'] = pd.to_datetime(d_load['full_date']).dt.date
rows = [tuple(row) for row in d_load.itertuples(index=False)]

execute_values(cursor, '''
    INSERT INTO "media".Dim_Date (date_key, full_date, year, quarter, month, month_name, decade)
    VALUES %s
''', rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.dim_date")

Loaded 204 rows into media.dim_date


##### Load DIM_MEDIA_TYPE (SCD1)¶

In [1589]:
cursor.execute('''
    DROP TABLE IF EXISTS "media".dim_media_type CASCADE;
    CREATE TABLE "media".dim_media_type (
        media_type_key INTEGER PRIMARY KEY,
        media_type     TEXT,
        category       TEXT,
        sub_category   TEXT
    );
''')
conn.commit()

rows = [tuple(row) for row in Dim_Media_Type.itertuples(index=False)]

execute_values(cursor, '''
    INSERT INTO "media".dim_media_type (media_type_key, media_type, category, sub_category)
    VALUES %s
''', rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.dim_media_type")

Loaded 5 rows into media.dim_media_type


##### Load DIM_GEOGRAPHY (SCD1)

In [1597]:
cursor.execute('''
    DROP TABLE IF EXISTS "media".dim_geography CASCADE;
    CREATE TABLE "media".dim_geography (
        geography_key INTEGER PRIMARY KEY,
        region_name   TEXT,
        region_level  TEXT,
        is_us_related BOOLEAN
    );
''')
conn.commit()

rows = [tuple(row) for row in Dim_Geography.itertuples(index=False)]

execute_values(cursor, '''
    INSERT INTO "media".dim_geography (geography_key, region_name, region_level, is_us_related)
    VALUES %s
''', rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.dim_geography")

Loaded 10 rows into media.dim_geography


##### Load DIM_SWITCH_REASON (SCD1)¶

In [1602]:
cursor.execute('''
    DROP TABLE IF EXISTS "media".dim_switch_reason CASCADE;
    CREATE TABLE "media".dim_switch_reason (
        reason_key          INTEGER PRIMARY KEY,
        primary_reason      TEXT,
        reason_category     TEXT,
        is_cost_related     BOOLEAN,
        is_content_related  BOOLEAN
    );
''')
conn.commit()

rows = [tuple(row) for row in Dim_Switch_Reason.itertuples(index=False)]

execute_values(cursor, '''
    INSERT INTO "media".dim_switch_reason
        (reason_key, primary_reason, reason_category,
         is_cost_related, is_content_related)
    VALUES %s
''', rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.dim_switch_reason")

Loaded 6 rows into media.dim_switch_reason


##### Load DIM_SUBSCRIPTION_PLAN (SCD2)¶

In [1609]:
cursor.execute('''
    DROP TABLE IF EXISTS "media".dim_subscription_plan CASCADE;
    CREATE TABLE "media".dim_subscription_plan (
        plan_key         INTEGER PRIMARY KEY,
        platform         TEXT,
        price            NUMERIC(6,2),
        price_tier       TEXT,
        effective_date   DATE,
        end_date         DATE,
        is_current       BOOLEAN
    );
''')
conn.commit()

dim_load = Dim_Subscription_Plan.copy()
dim_load['effective_date'] = pd.to_datetime(dim_load['effective_date']).dt.date
dim_load['end_date']       = pd.to_datetime(dim_load['end_date']).dt.date
dim_load['price_tier']     = dim_load['price_tier'].astype(str)

rows = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in dim_load[[
        'plan_key', 'platform', 'price', 'price_tier',
        'effective_date', 'end_date', 'is_current'
    ]].itertuples(index=False)
]

execute_values(cursor, '''
    INSERT INTO "media".dim_subscription_plan
        (plan_key, platform, price, price_tier, effective_date, end_date, is_current)
    VALUES %s
''', rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.dim_subscription_plan")

Loaded 27 rows into media.dim_subscription_plan


##### Load DIM_PLATFORM (SCD3)¶

In [1613]:
cursor.execute('''
    DROP TABLE IF EXISTS "media".dim_platform CASCADE;
    CREATE TABLE "media".dim_platform (
        platform_key                INTEGER PRIMARY KEY,
        platform                    TEXT,
        current_parent_company      TEXT,
        previous_parent_company     TEXT,
        current_business_model      TEXT,
        previous_business_model     TEXT,
        parent_company_change_date  DATE,
        content_focus               TEXT,
        launch_year                 INTEGER,
        is_digital                  BOOLEAN,
        media_sector                TEXT
    );
''')
conn.commit()

dim_p = Dim_Platform[[
    'platform_key', 'platform', 'current_parent_company', 'previous_parent_company',
    'current_business_model', 'previous_business_model',
    'parent_company_change_date',
    'content_focus', 'launch_year', 'is_digital', 'media_sector'
]].copy()

dim_p['parent_company_change_date'] = pd.to_datetime(dim_p['parent_company_change_date']).dt.date

rows = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in dim_p.itertuples(index=False)
]
execute_values(cursor, '''
    INSERT INTO "media".dim_platform
        (platform_key, platform,
         current_parent_company, previous_parent_company,
         current_business_model, previous_business_model,
         parent_company_change_date,
         content_focus, launch_year, is_digital, media_sector)
    VALUES %s
''', rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.dim_platform")

Loaded 14 rows into media.dim_platform


##### Load FACT_SUBSCRIPTION_PRICING into PostgreSQL

In [1618]:
cursor.execute("""
    DROP TABLE IF EXISTS "media".fact_subscription_pricing;
    CREATE TABLE "media".fact_subscription_pricing (
        pricing_key          SERIAL PRIMARY KEY,
        date_key             INTEGER REFERENCES "media".dim_date(date_key),
        platform_key         INTEGER REFERENCES "media".dim_platform(platform_key),
        plan_key             INTEGER REFERENCES "media".dim_subscription_plan(plan_key),
        price                NUMERIC(6,2),
        price_change_mom     NUMERIC(6,2),
        cumulative_increase  NUMERIC(6,2)
    );
""")
conn.commit()

fact_load = Fact_Subscription_Pricing[[
    'date_key', 'platform_key', 'plan_key',
    'price', 'price_change_mom', 'cumulative_increase'
]].copy()

rows = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in fact_load.itertuples(index=False)
]

execute_values(cursor, """
    INSERT INTO "media".fact_subscription_pricing
        (date_key, platform_key, plan_key,
         price, price_change_mom, cumulative_increase)
    VALUES %s
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.fact_subscription_pricing")

Loaded 777 rows into media.fact_subscription_pricing


##### Load FACT_MEDIA_PERFORMANCE into PostgreSQL

In [1621]:
cursor.execute("""
    DROP TABLE IF EXISTS "media".fact_media_performance;
    CREATE TABLE "media".fact_media_performance (
        perf_key                SERIAL PRIMARY KEY,
        date_key                INTEGER REFERENCES "media".dim_date(date_key),
        platform_key            INTEGER REFERENCES "media".dim_platform(platform_key),
        media_type_key          INTEGER REFERENCES "media".dim_media_type(media_type_key),
        subscribers_millions    NUMERIC(10,3),
        viewership_millions     NUMERIC(10,3),
        revenue_usd_millions    NUMERIC(12,2),
        yoy_growth_pct          NUMERIC(6,2),
        cumulative_subscribers  NUMERIC(10,2)
    );
""")
conn.commit()

fact_load = Fact_Media_Performance[[
    'date_key', 'platform_key', 'media_type_key',
    'subscribers_millions', 'viewership_millions', 'revenue_usd_millions',
    'yoy_growth_pct', 'cumulative_subscribers'
]].copy()

rows = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in fact_load.itertuples(index=False)
]

execute_values(cursor, """
    INSERT INTO "media".fact_media_performance
        (date_key, platform_key, media_type_key,
         subscribers_millions, viewership_millions, revenue_usd_millions,
         yoy_growth_pct, cumulative_subscribers)
    VALUES %s
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.fact_media_performance")

Loaded 1533 rows into media.fact_media_performance


##### Load FACT_ENGAGEMENT into PostgreSQL

In [1627]:
cursor.execute("""
    DROP TABLE IF EXISTS "media".fact_engagement;
    CREATE TABLE "media".fact_engagement (
        engagement_key                 SERIAL PRIMARY KEY,
        date_key                       INTEGER REFERENCES "media".dim_date(date_key),
        platform_key                   INTEGER REFERENCES "media".dim_platform(platform_key),
        geo_key                        INTEGER REFERENCES "media".dim_geography(geography_key),
        switch_reason_key              INTEGER REFERENCES "media".dim_switch_reason(reason_key),
        avg_hours_per_user             NUMERIC(6,2),
        monthly_active_users_millions  NUMERIC(10,3),
        avg_sessions_per_month         NUMERIC(6,2),
        retention_rate_pct             NUMERIC(6,2)
    );
""")
conn.commit()

fact_load = Fact_Engagement[[
    'date_key', 'platform_key', 'geo_key', 'switch_reason_key',
    'avg_hours_per_user', 'monthly_active_users_millions',
    'avg_sessions_per_month', 'retention_rate_pct'
]].copy()

rows = [
    tuple(None if pd.isna(x) else x for x in row)
    for row in fact_load.itertuples(index=False)
]

execute_values(cursor, """
    INSERT INTO "media".fact_engagement
        (date_key, platform_key, geo_key, switch_reason_key,
         avg_hours_per_user, monthly_active_users_millions,
         avg_sessions_per_month, retention_rate_pct)
    VALUES %s
""", rows)
conn.commit()
print(f"Loaded {len(rows)} rows into media.fact_engagement")

Loaded 4080 rows into media.fact_engagement


> All 3 facts + 6 dims loaded to `media` schema.  
> - `fact_subscription_pricing` — 777 rows (platform \u00d7 month)  
> - `fact_media_performance` — streaming + traditional rows (platform \u00d7 month \u00d7 media_type)  
> - `fact_engagement` — platform \u00d7 month \u00d7 region rows
